# 訓練診斷

[完整圖文教材](https://birdhackor.github.io/learn_to_yolo/lessons/02-diagnostics/)

先執行下一格的環境格，再閱讀、執行最後的完整實驗。各節互相獨立，不需要先跑其他節。

最後一格存著這段程式在 CPU 上實際執行的輸出；執行環境與其他實驗紀錄見網站的〈驗證範圍與後續實驗〉。範例不用預訓練權重，也不需要下載資料。

In [ ]:
# 全新 runtime 可由此格開始。只取得小型程式，不自動下載 LFS 資料。
import importlib.metadata
import os
from pathlib import Path
import subprocess
import sys

REF = 'lessons-v0.4.0'
base = Path("/content") if Path("/content").is_dir() else Path.cwd()
repository = base / ("learn_to_yolo_" + REF.replace(".", "_").replace("-", "_"))
if not repository.exists():
    environment = os.environ.copy()
    environment["GIT_LFS_SKIP_SMUDGE"] = "1"
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REF,
                    "https://github.com/birdhackor/learn_to_yolo.git", str(repository)],
                   env=environment, check=True)
actual_ref = subprocess.check_output(
    ["git", "-C", str(repository), "describe", "--tags", "--exact-match"], text=True
).strip()
if actual_ref != REF:
    raise RuntimeError("程式版本不同，請使用新的 runtime 或移除舊 clone 後重試。")

# 各節實驗都在 CPU 上執行，執行紀錄用的是 PyTorch 2.9.1。已經是 2.9.1 就沿用（CPU 或 CUDA 版都可以）；
# 其他版本會改裝成 2.9.1 的 CPU 版，之後這個工作階段就用不到 GPU，但各節實驗本來就不需要。
try:
    torch_version = importlib.metadata.version("torch").split("+")[0]
except importlib.metadata.PackageNotFoundError:
    torch_version = None
if torch_version != "2.9.1":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch==2.9.1",
                    "--index-url", "https://download.pytorch.org/whl/cpu"], check=True)
    if "torch" in sys.modules:
        raise RuntimeError("PyTorch 已更新：請從選單重新啟動工作階段，再由第一格執行。")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                *['numpy==2.3.5', 'pillow==12.0.0', 'matplotlib==3.10.7']], check=True)
os.chdir(repository)
sys.path.insert(0, str(repository))
print("固定教材版本：", REF)
print("目前目錄：", repository)


## 本節可修改的完整實驗

先預測結果，再執行；確認輸出後，試做網頁的自主練習。

In [1]:
"""Controlled failures: disconnected gradient, invalid label, spurious correlation."""
import torch
from torch import nn


def main():
    torch.manual_seed(7)
    torch.set_num_threads(2)
    body, head = nn.Linear(2, 3), nn.Linear(3, 2)
    optimizer = torch.optim.SGD(list(body.parameters()) + list(head.parameters()), lr=0.2)
    x, labels = torch.tensor([[-0.2, -1.0], [0.2, 1.0]]), torch.tensor([0, 1])
    optimizer.zero_grad(set_to_none=True)
    loss = nn.functional.cross_entropy(head(body(x).detach()), labels)
    loss.backward()
    assert body.weight.grad is None and head.weight.grad is not None
    print("broken graph: body.weight.grad=None, head.weight.grad exists")
    optimizer.zero_grad(set_to_none=True)
    before = body.weight.detach().clone()
    loss = nn.functional.cross_entropy(head(body(x)), labels)
    loss.backward()
    assert body.weight.grad is not None and body.weight.grad.abs().sum() > 0
    optimizer.step()
    assert not torch.equal(before, body.weight)
    print("repaired graph: body gradient nonzero and body parameter changed")

    bad_labels = torch.tensor([0, 2])
    valid_labels = ((bad_labels >= 0) & (bad_labels < 2)).all()
    assert not valid_labels
    print("label preflight: [0, 2] invalid for two classes; expected IDs 0 or 1")

    # Feature 0 is a weak true cue; feature 1 is a strong incidental corner cue.
    train_x, train_y = x.repeat(4, 1), labels.repeat(4)
    validation_x = torch.tensor([[-0.2, 1.0], [0.2, -1.0]]).repeat(4, 1)
    validation_y = labels.repeat(4)
    model = nn.Linear(2, 2, bias=False)
    with torch.no_grad():
        model.weight.zero_()
    optimizer = torch.optim.SGD(model.parameters(), lr=0.2)
    for step in range(20):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        logits = model(train_x)
        loss = nn.functional.cross_entropy(logits, train_y)
        loss.backward()
        assert torch.isfinite(model.weight.grad).all()
        optimizer.step()
        model.eval()
        with torch.no_grad():
            train_loss = nn.functional.cross_entropy(model(train_x), train_y).item()
            val_loss = nn.functional.cross_entropy(model(validation_x), validation_y).item()
        if step in (0, 9, 19):
            print(f"step={step}: train_loss={train_loss:.4f}, validation_loss={val_loss:.4f}")
    with torch.no_grad():
        train_acc = (model(train_x).argmax(1) == train_y).float().mean().item()
        val_acc = (model(validation_x).argmax(1) == validation_y).float().mean().item()
    print(f"train_accuracy={train_acc:.2f}; validation_accuracy={val_acc:.2f}")
    print(f"learned weights={model.weight.detach().tolist()}")
    assert train_acc == 1.0 and val_acc == 0.0
    print("Synthetic distribution shift demonstrates: tiny-set overfit is not generalization.")


if __name__ == "__main__":
    main()


broken graph: body.weight.grad=None, head.weight.grad exists
repaired graph: body gradient nonzero and body parameter changed
label preflight: [0, 2] invalid for two classes; expected IDs 0 or 1
step=0: train_loss=0.5945, validation_loss=0.7937
step=9: train_loss=0.2244, validation_loss=1.5204
step=19: train_loss=0.1236, validation_loss=2.0153
train_accuracy=1.00; validation_accuracy=0.00
learned weights=[[-0.19503174722194672, -0.9751587510108948], [0.19503173232078552, 0.9751587510108948]]
Synthetic distribution shift demonstrates: tiny-set overfit is not generalization.
